# Spelling Correction with the Noisy Channel Model

## Task 1: Data Loading and Preprocessing

First, you need to load the dataset and preprocess it to extract the necessary information. The dataset contains text with misspelled words tagged. Your goal is to create a vocabulary of correct words and a dictionary of error-correction pairs.

Dataset Link: https://titan.dcs.bbk.ac.uk/~roger/holbrook-tagged.dat

In [ ]:
from google.colab import files
import re
from collections import Counter

# TODO: Upload holbrook-tagged-1.dat
# Hint: files.upload() returns a dict of {filename: bytes}; the key is your file's name
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

# TODO: Read the file's text
# Hint: the dataset has special (non-UTF-8) characters, so open it with encoding='latin-1'
with open(data_path, 'r', encoding='latin-1') as f:
  text = f.read()

# TODO: Write a regex pattern to extract (correct_word, wrong_word) pairs
# Hint: errors in the dataset look like <ERR targ=correct> wrong </ERR> (no quotes around targ,
#       and there's whitespace around the misspelling)
pattern = r'<ERR targ=(\S+)>([^<]+)</ERR>'
matches = re.findall(pattern, text)
print(f"Total error annotations found: {len(matches)}")

# TODO: Build a vocabulary Counter from every correct/wrong word found
# Hint: lowercase and .strip() each word before counting, and count both
#       the correct spelling and the misspelling so both are in vocab
vocab = Counter()
for correct, wrong in matches:
    vocab[correct.strip().lower()] += 1
    vocab[wrong.strip().lower()] += 1

print(f"Total error annotations found: {len(matches)}")
print(f"Vocabulary size: {len(vocab)}")
print("Example P(c):", list(vocab.items())[:10])

## Task 2: Building the Noisy Channel Model

The noisy channel model for spelling correction is based on Bayes' theorem. We want to find the most probable correction `c` for a given (potentially misspelled) word `w`:

$$ \text{argmax}_c P(c|w) = \text{argmax}_c \frac{P(w|c)P(c)}{P(w)} $$

Since `P(w)` is the same for all candidate corrections, we can simplify this to:

$$ \text{argmax}_c P(w|c)P(c) $$

Here:
- `P(c)` is the **language model**: the probability of the correction `c` appearing in the text. We can estimate this from the word frequencies in our corpus.
- `P(w|c)` is the **error model**: the probability that the writer produced the word `w` when they intended to write `c`. We'll estimate this based on the edit distance between `w` and `c`.

In [ ]:
def P_c(c):
    # language model probability P(c): frequency of c in vocab, divided by total count of all words in vocab
    return vocab[c] / sum(vocab.values())

def P_w_given_c(w, c):
    # error model P(w|c) based on inverse edit distance
    return 1 / (edit_distance(w, c) + 1)

def noisy_channel_score(w, c):
    # combine P(w|c) and P(c) as in the formula above
    return P_w_given_c(w, c) * P_c(c)

## Task 3: Edit Distance and Candidate Generation

To find potential corrections for a misspelled word, we need to generate a set of 'candidate' words that are close to the original. A common way to do this is to generate all words that are a small 'edit distance' away.

In [ ]:
# Edit Distance function (Levenshtein distance)
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],    # deletion
                    dp[i][j - 1],    # insertion
                    dp[i - 1][j - 1] # substitution
                )
    return dp[m][n]

def calculate_edit_distance(s1, s2):

    m, n = len(s1), len(s2)

    # Initialize the DP matrix with zeros. Dimensions are (m+1) x (n+1).
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # --- Step 2: Initialization ---
    # Cost of deletions (transforming s1 to empty string)
    for i in range(m + 1):
        dp[i, 0] = i

    # Cost of insertions (transforming empty string to s2)
    for j in range(n + 1):
        dp[0, j] = j

    # --- Step 3: Fill the Matrix ---
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # If characters are the same, cost is 0, take diagonal value.
            # Note: string indices are i-1 and j-1 because DP table is 1-indexed.
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1 # Cost for substitution

            # Calculate costs for each operation
            deletion_cost = dp[i - 1, j] + 1
            insertion_cost = dp[i, j - 1] + 1
            substitution_cost = dp[i - 1, j - 1] + cost

            # The value of dp[i,j] is the minimum of these three costs
            dp[i, j] = min(deletion_cost, insertion_cost, substitution_cost)

    # --- Step 4: Final Result ---
    edit_distance = dp[m, n]

    return edit_distance, dp


def generate_candidates(w, max_dist=2):
    # return the set of vocab words within max_dist edit distance of w
    return {word for word in vocab if edit_distance(w, word) <= max_dist}


## Task 4: Create the Spelling Corrector

Now, combine the components to create your spelling corrector. For a given word, you will generate candidates and then score them using the noisy channel model to find the best correction.

In [ ]:
def correct_word(w):
    w = w.lower()
    # if w is already a known/correct word, just return it
    if w in vocab:
        return w
    # otherwise, generate_candidates(w); if there are none, fall back to returning w
    candidates = generate_candidates(w)
    if not candidates:
        return w
    return max(candidates, key=lambda c: noisy_channel_score(w, c))


def correct_sentence(sentence):
    # apply correct_word to every word in the sentence and join back with spaces
    return ' '.join(correct_word(w) for w in sentence.split())


# Test on given sentences
test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

for sent in test_sentences:
    print("Original:", sent)
    print("Corrected:", correct_sentence(sent))
    print()

## Task 5: Comparison with a Standard Library

Finally, let's compare your model's performance with a well-established spell-checking library, `pyspellchecker`. This will give you a baseline to understand how your model performs and where it could be improved.

In [ ]:
import numpy as np
!pip install pyspellchecker
from spellchecker import SpellChecker

spell = SpellChecker()

test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

for sent in test_sentences:
    words = sent.split()
    print("\nSentence:", sent)

    # our model's correction for each word (only touch words not already in vocab)
    our_correction = [w if w.lower() in vocab else correct_word(w) for w in words]
    print("Our Model Correction:", ' '.join(our_correction))

    # pyspellchecker's correction for each word
    pyspell_correction = [spell.correction(w) or w for w in words]
    print("Pyspellchecker Correction:", ' '.join(pyspell_correction))